In [3]:
from fastapi import FastAPI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np

app = FastAPI()

# Variables globales
movies_with_genres2 = None
similarity_matrix = None


    # Cargar los datasets
movies_api1 = pd.read_csv("recomendaciones/recomendacion_api.csv")
genres_data1 = pd.read_csv("recomendaciones/genres_api.csv")

    # Asegurarse de que ambas columnas tengan el mismo tipo
movies_api1["id"] = movies_api1["id"].astype(str)
genres_data1["id_original"] = genres_data1["id_original"].astype(str)

    # Procesar géneros para asegurar que no se repitan
movies_with_genres1 = pd.merge(movies_api1, genres_data1, left_on="id", right_on="id_original", how="left")

    # Filtrar y agrupar datos
movies_with_genres2 = movies_with_genres1.sample(n=10000, random_state=42)
movies_with_genres2 = movies_with_genres2.groupby("id_original").agg({
        "title": "first",
        "name": lambda x: " ".join(set(x.dropna())),
        "vote_average": "first",
        "popularity": "first"
    }).reset_index()

    # Combinar texto y géneros en una nueva columna 'combined_features'
movies_with_genres2["combined_features"] = (
        movies_with_genres2["title"].fillna("") + " " +
        movies_with_genres2["name"].fillna("")
    )

    # Vectorizar el texto combinado usando TF-IDF
tfidf = TfidfVectorizer(stop_words="english", max_features=5000)
tfidf_matrix = tfidf.fit_transform(movies_with_genres2["combined_features"])

    # Normalizar las puntuaciones numéricas
scaler = MinMaxScaler()
movies_with_genres2[["vote_average", "popularity"]] = scaler.fit_transform(
    movies_with_genres2[["vote_average", "popularity"]]
    )

    # Reducir el tipo de datos para ahorrar memoria
movies_with_genres2 = movies_with_genres2.astype({'vote_average': 'float16', 'popularity': 'float16'})

    # Calcular la matriz de similitud usando Cosine Similarity
numerical_features = movies_with_genres2[["vote_average", "popularity"]].values
final_features = np.hstack((tfidf_matrix.toarray(), numerical_features))
similarity_matrix = cosine_similarity(final_features)

@app.get("/recomendacion/{titulo}")
def recomendacion1(titulo: str):
    global movies_with_genres2, similarity_matrix

    try:
        # Verificar que el título esté en el dataset
        if titulo not in movies_with_genres2["title"].values:
            return {"mensaje": f"La película '{titulo}' no se encuentra en el dataset."}

        # Obtener el índice de la película dada
        idx = movies_with_genres2[movies_with_genres2["title"] == titulo].index[0]

        # Ordenar películas por similitud
        similarity_scores = list(enumerate(similarity_matrix[idx]))
        similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

        # Obtener las 5 películas más similares (excluyendo duplicados y la misma película)
        seen_titles = set([titulo])
        top_movies = []
        for i, score in similarity_scores[1:]:
            movie_title = movies_with_genres2.iloc[i]["title"]
            if movie_title not in seen_titles:
                seen_titles.add(movie_title)
                top_movies.append(movie_title)
            if len(top_movies) == 5:
                break

        return {"recomendaciones": top_movies}
    except Exception as e:
        return {"error": f"Ocurrió un error: {str(e)}"}


In [4]:
movies_with_genres2.head

C:\Python\Lib\site-packages\pandas\io\formats\format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


<bound method NDFrame.head of      id_original                                title       name  \
0         100010                       Flight Command      Drama   
1         100017                              Hounded      Drama   
2          10005  Behind Enemy Lines II: Axis of Evil   Thriller   
3         100110                       Survival Quest  Adventure   
4          10013                Peggy Sue Got Married      Drama   
...          ...                                  ...        ...   
8894        9987                            Headspace   Thriller   
8895       99898                   Man on a Tightrope   Thriller   
8896        9990           Walking Tall: Lone Justice   Thriller   
8897        9992            Arthur and the Invisibles     Family   
8898        9994            The Great Mouse Detective     Family   

      vote_average  popularity                             combined_features  
0         0.600098    0.001405                          Flight Command Dra

In [5]:
recomendacion1("Hounded")

{'recomendaciones': ['Buitenspel',
  'Sansho the Bailiff',
  'On the Other Side',
  'You and Me',
  'Sebastiane']}